In [1]:
import json
import os
from pathlib import Path
from urllib.parse import quote as url_encode
import hashlib
from shapely.geometry import LineString
import zipfile

In [2]:
trips = []
for ut in os.listdir("trips_cache"):
    if os.path.isdir(os.path.join("trips_cache", ut)):
        for line in os.listdir(os.path.join("trips_cache", ut)):
            for dir in os.listdir(os.path.join("trips_cache", ut, line)):
                for file in os.listdir(os.path.join("trips_cache", ut, line, dir)):
                    with open(os.path.join("trips_cache", ut, line, dir, file), "r") as f:
                        data = json.load(f)
                        trips.append(data)

In [3]:
def hash_linestring(line_string: list) -> str:
    line_string = LineString(line_string)
    line_string = line_string.normalize()
    
    wkb = line_string.wkb
    
    return hashlib.sha256(wkb).hexdigest()

In [4]:

agency = ("UNIR", "UNIR: Mobilidade da Area Metropolitan do Porto", "https://www.unirmobilidade.pt/", "Europe/Lisbon", "pt")
calendar = [("UTEIS", 1, 1, 1, 1, 1, 0, 0, "20260101", "20301231")]
routes = {}
stops = {}
stop_times = []
trips_m = {}
shapes = {}

for trip in trips:
    if trip["cod"] not in routes:
        route = (trip["cod"], trip["cod"], f"{trip['origem']} - {trip['destino']}", 700)
        routes[trip["cod"]] = route

    shape_id_hash = hash_linestring(trip["geometry"]["coordinates"])

    if shape_id_hash in shapes:
        shape_id = shapes[shape_id_hash]["id"]
        if shape_id != trip["cod_netex"].replace(":","_"):
            exception = 1
            while shape_id+f"_{exception}" in shapes:
                 exception += 1
            print(f"Collision detected for shape_id {shape_id}. Appending exception number to resolve: {shape_id}_{exception}")
            shape_id += f"_{exception}"
            shapes[shape_id] = {"id":shape_id,"entries": []}
            for i, point in enumerate(trip["geometry"]["coordinates"]):
                entry = (shape_id, float(point[1]), float(point[0]), i+1)
                shapes[shape_id]["entries"].append(entry)
            
    else :
        shape_id = trip["cod_netex"].replace(":","_")
        shapes[shape_id_hash] = {"id":shape_id,"entries": []}
        for i, point in enumerate(trip["geometry"]["coordinates"]):
            entry = (shape_id, float(point[1]), float(point[0]), i+1)
            shapes[shape_id_hash]["entries"].append(entry)
        
    if trip["servico"].replace(":","_") not in trips_m:
        entry = (trip["cod"], trip["sentido"], "UTEIS", trip["servico"].replace(":","_"), trip["destino"], 1, "", shape_id)
        trips_m[trip["servico"].replace(":","_")] = entry

    for stop in trip["horario"]:
        stop_id = stop["cod_paragem"].replace(":","_")
        if not stop_id in stops:
            entry = (stop_id,stop_id, stop["desig_paragem"].replace(",", ";"),stop["lat"], stop["lon"],stop["zona_andante"],f"https://paragens.amp.pt/web/qihoras/pages/stop.html?id={url_encode(stop['cod_paragem'])}")
            stops[stop_id] = entry
        stop_time = (trip["servico"].replace(":","_"), stop["tempo"], stop["tempo"], stop_id, stop["ordem"], "")
        stop_times.append(stop_time)

Collision detected for shape_id 9028_0_1. Appending exception number to resolve: 9028_0_1_1


In [ ]:
Path("gtfs").mkdir(exist_ok=True)

with open("gtfs/agency.txt", "w") as f:
    f.write("agency_id,agency_name,agency_url,agency_timezone,agency_lang\n")
    f.write(",".join(agency) + "\n")

with open("gtfs/routes.txt", "w") as f:
    f.write("route_id,route_short_name,route_long_name,route_type\n")
    for route in routes.values():
        f.write(",".join(map(str, route)) + "\n")

with open("gtfs/stops.txt", "w") as f:
    f.write("stop_id,stop_code,stop_name,stop_lat,stop_lon,zone_id,stop_url\n")
    for stop in stops.values():
        f.write(",".join(map(str, stop)) + "\n")

with open("gtfs/stop_times.txt", "w") as f:
    f.write("trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign\n")
    for stop_time in stop_times:
        f.write(",".join(map(str, stop_time)) + "\n")

with open("gtfs/trips.txt", "w") as f:
    f.write("route_id,direction_id,service_id,trip_id,trip_headsign,wheelchair_accessible,block_id,shape_id\n")
    for trip in trips_m.values():
        f.write(",".join(map(str, trip)) + "\n")

with open("gtfs/shapes.txt", "w") as f:
    f.write("shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence\n")
    for shape in shapes.values():
        for entry in shape["entries"]:
            f.write(",".join(map(str, entry)) + "\n")

with open("gtfs/calendar.txt", "w") as f:
    f.write("service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date\n")
    for service in calendar:
        f.write(",".join(map(str, service)) + "\n")

with zipfile.ZipFile("gtfs_unir.zip", "w") as zipf:
    for file in os.listdir("gtfs"):
        zipf.write(os.path.join("gtfs", file), file)